# AI Documentary Generator — Free on Kaggle T4

Generates long-form documentary videos using **zero-cost** tools:
- **SDXL** for high-quality images
- **Stable Video Diffusion** (optional) for video motion
- **Edge-TTS** for narration (no API key)
- **FFmpeg** for final assembly

**Setup:** Settings → Accelerator → `GPU T4 x2`  |  Internet → `On`

In [ ]:
# Cell 1: Install dependencies (~3 min first run)

import subprocess, sys, os

os.environ.pop('TF_CPP_MIN_LOG_LEVEL', None)
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q diffusers transformers accelerate safetensors
!pip install -q pillow edge-tts imageio[ffmpeg] soundfile
!apt-get install -qq ffmpeg 2>/dev/null

print('Dependencies installed!')

In [ ]:
# Cell 2: Verify environment

import torch
import platform

print(f"Python {platform.python_version()}  |  PyTorch {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu}  |  VRAM: {vram:.1f} GB")

import subprocess
r = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print('FFmpeg:', 'OK' if r.returncode == 0 else 'MISSING')

---
## CONFIGURE YOUR DOCUMENTARY

Edit the **topic** and **settings** in Cell 3 below, then run everything.

In [ ]:
# Cell 3: Configuration — EDIT THIS

TOPIC = "The History of Artificial Intelligence"  # ← Your documentary topic
NUM_SCENES = 6                                      # ← 4-10 scenes
WIDTH, HEIGHT = 1024, 576                           # ← 16:9 resolution
ENABLE_MOTION = False       # True = SVD video clips (slower), False = Ken Burns
VOICE = "en-US-GuyNeural"   # Edge-TTS voice

OUTPUT_DIR = "/kaggle/working/output"
print(f"Topic: {TOPIC}")
print(f"Scenes: {NUM_SCENES}  |  Resolution: {WIDTH}x{HEIGHT}  |  Motion: {ENABLE_MOTION}")

In [ ]:
# Cell 4: Script Generator

import json, os, textwrap

def generate_script(topic, num_scenes):
    templates = [
        {
            "title": f"Introduction to {topic}",
            "narration": f"Welcome to our documentary exploring {topic}. "
                         f"This fascinating subject has shaped our world in countless ways. "
                         f"Join us as we journey through its origins, evolution, and impact. "
                         f"From humble beginnings to groundbreaking discoveries, this is the story of {topic}.",
            "image_prompt": f"Cinematic wide shot of a dramatic landscape at sunrise, representing the dawn of {topic}, "
                           f"golden hour lighting, professional photography, 8K resolution, National Geographic style",
            "duration": 12,
        },
        {
            "title": f"The Origins of {topic}",
            "narration": f"The roots of {topic} stretch back further than many realize. "
                        f"Early pioneers laid the groundwork for what would become a revolutionary field. "
                        f"Their vision and determination set the stage for transformative change.",
            "image_prompt": f"Historical photograph style showing early laboratory or workshop related to {topic}, "
                           f"vintage aesthetic, warm sepia tones, scientists working with early equipment, documentary style",
            "duration": 10,
        },
        {
            "title": f"Key Breakthroughs in {topic}",
            "narration": f"Several key breakthroughs propelled {topic} forward. "
                        f"Each discovery built upon the last, creating a cascade of innovation. "
                        f"These moments changed everything and opened new frontiers of possibility.",
            "image_prompt": f"Dramatic visualization of a major scientific breakthrough, glowing elements, "
                           f"particles of light, technological advancement, cinematic lighting, epic scale",
            "duration": 10,
        },
        {
            "title": f"How {topic} Affects Our World",
            "narration": f"Today, {topic} touches nearly every aspect of modern life. "
                        f"From healthcare to communication, its influence is everywhere. "
                        f"We see its impact in the tools we use and the world we build.",
            "image_prompt": f"Modern cityscape showing technology and innovation connected to {topic}, "
                           f"futuristic architecture, digital networks visualized as light trails, cyberpunk aesthetic",
            "duration": 10,
        },
        {
            "title": f"Challenges Facing {topic}",
            "narration": f"Despite its progress, {topic} faces significant challenges. "
                        f"Ethical questions, technical limitations, and societal concerns all demand attention. "
                        f"How we address these will shape the future of the field.",
            "image_prompt": f"Contemplative scene showing a person facing a massive digital interface, "
                           f"blue and purple neon lighting, questions and symbols floating, thoughtful mood",
            "duration": 10,
        },
        {
            "title": f"The Future of {topic}",
            "narration": f"Looking ahead, the future of {topic} is filled with possibility. "
                        f"Emerging trends point toward even greater integration into our lives. "
                        f"The journey is far from over — the best chapters may still be unwritten.",
            "image_prompt": f"Utopian future scene representing the bright future of {topic}, "
                           f"floating cities, holographic displays, diverse people working together, optimistic lighting",
            "duration": 12,
        },
    ]
    
    scenes = []
    for i in range(num_scenes):
        t = templates[i % len(templates)]
        scenes.append({
            "scene_id": i + 1,
            "scene_title": t["title"],
            "narration": t["narration"],
            "image_prompt": t["image_prompt"],
            "duration_sec": t["duration"],
        })
    
    return {"topic": topic, "scenes": scenes}

script = generate_script(TOPIC, NUM_SCENES)
os.makedirs(f"{OUTPUT_DIR}/script", exist_ok=True)
with open(f"{OUTPUT_DIR}/script/script.json", "w") as f:
    json.dump(script, f, indent=2)

print(f"Script generated: {NUM_SCENES} scenes")
for s in script["scenes"]:
    print(f"  {s['scene_id']}. {s['scene_title']}")

In [ ]:
# Cell 5: Generate Images with SDXL on T4

from diffusers import DiffusionPipeline
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading SDXL on {device}...")

pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    use_safetensors=True,
)
if device == "cuda":
    pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()

image_dir = f"{OUTPUT_DIR}/images"
os.makedirs(image_dir, exist_ok=True)

negative = "blurry, low quality, distorted, ugly, deformed, text, watermark"
image_paths = []

for scene in script["scenes"]:
    sid = scene["scene_id"]
    prompt = scene["image_prompt"]
    out = f"{image_dir}/scene_{sid:02d}.png"
    
    print(f"[{sid}/{len(script['scenes'])]} Generating... {prompt[:50]}...")
    images = pipe(
        prompt=prompt,
        negative_prompt=negative,
        width=WIDTH, height=HEIGHT,
        num_inference_steps=25,
        guidance_scale=7.0,
    ).images
    images[0].save(out, quality=95)
    image_paths.append(out)
    print(f"  Saved: {out}")

del pipe
if device == "cuda":
    torch.cuda.empty_cache()

print(f"\nAll {len(image_paths)} images generated!")

**Optional:** If you want video motion (SCENE CLIPS), run Cell 6.
Otherwise the pipeline will use Ken Burns effect on still images. Skip to Cell 7.

In [ ]:
# Cell 6 (OPTIONAL): Generate Video Clips with SVD
# Only run this if ENABLE_MOTION = True, and you have time (~2 min per clip)

if ENABLE_MOTION:
    from diffusers import StableVideoDiffusionPipeline
    from PIL import Image
    import imageio
    
    print("Loading SVD on GPU...")
    svd = StableVideoDiffusionPipeline.from_pretrained(
        "stabilityai/stable-video-diffusion-img2vid-xt",
        torch_dtype=torch.float16,
        variant="fp16",
    )
    svd.enable_model_cpu_offload()
    svd.enable_vae_slicing()
    
    video_dir = f"{OUTPUT_DIR}/videos"
    os.makedirs(video_dir, exist_ok=True)
    video_paths = []
    
    for i, img_path in enumerate(image_paths):
        duration = script["scenes"][i]["duration_sec"] if i < len(script["scenes"]) else 10
        n_frames = min(int(duration * 12), 25)
        out = f"{video_dir}/clip_{i+1:02d}.mp4"
        
        print(f"[{i+1}/{len(image_paths)]} Generating clip ({n_frames} frames)...")
        image = Image.open(img_path).convert("RGB")
        frames = svd(
            image,
            decode_chunk_size=8,
            num_frames=n_frames,
            motion_bucket_id=127,
            noise_aug_strength=0.02,
        ).frames[0]
        
        writer = imageio.get_writer(out, fps=12, codec="libx264")
        for frame in frames:
            writer.append_data(frame)
        writer.close()
        video_paths.append(out)
        print(f"  Saved: {out}")
    
    del svd
    torch.cuda.empty_cache()
    print(f"\n{len(video_paths)} video clips generated!")
else:
    print("Skipping (ENABLE_MOTION = False). Using still images with Ken Burns.")
    video_paths = image_paths

In [ ]:
# Cell 7: Generate Narration Audio (Edge-TTS — free, no API key)

import edge_tts
import asyncio

async def generate_audio(scenes):
    audio_dir = f"{OUTPUT_DIR}/audio"
    os.makedirs(audio_dir, exist_ok=True)
    paths = []
    for scene in scenes:
        out = f"{audio_dir}/scene_{scene['scene_id']:02d}.mp3"
        print(f"  TTS: Scene {scene['scene_id']}...")
        communicate = edge_tts.Communicate(scene["narration"], VOICE)
        await communicate.save(out)
        paths.append(out)
    return paths

print(f"Generating narration ({VOICE})...")
audio_paths = await generate_audio(script["scenes"])
print(f"{len(audio_paths)} audio files generated!")

In [ ]:
# Cell 8: Assemble Final Documentary with FFmpeg

import subprocess
from pathlib import Path

final_dir = f"{OUTPUT_DIR}/final"
os.makedirs(final_dir, exist_ok=True)
temp_dir = f"{OUTPUT_DIR}/temp"
os.makedirs(temp_dir, exist_ok=True)

scenes = script["scenes"]
res = f"{WIDTH}:{HEIGHT}"
fps = 24

# Build scene video segments
segment_files = []

for i in range(len(scenes)):
    img = image_paths[i] if not ENABLE_MOTION else video_paths[i]
    aud = audio_paths[i]
    dur = scenes[i]["duration_sec"]
    out = f"{temp_dir}/seg_{i:03d}.mp4"
    
    if ENABLE_MOTION and str(img).endswith('.mp4'):
        # Use video clip directly
        subprocess.run([
            "ffmpeg", "-y",
            "-i", img,
            "-i", aud,
            "-c:v", "copy",
            "-c:a", "aac",
            "-shortest",
            out
        ], check=True, capture_output=True)
    else:
        # Ken Burns effect on still image
        zoom = 1.02
        subprocess.run([
            "ffmpeg", "-y",
            "-loop", "1",
            "-i", img,
            "-i", aud,
            "-c:v", "libx264",
            "-t", str(dur),
            "-pix_fmt", "yuv420p",
            "-vf", (
                f"scale={WIDTH}:{HEIGHT}:force_original_aspect_ratio=decrease,"
                f"pad={WIDTH}:{HEIGHT}:(ow-iw)/2:(oh-ih)/2,"
                f"zoompan=z='min(zoom+0.0005,{zoom})':d={int(dur*fps)}:s={WIDTH}x{HEIGHT}:fps={fps}"
            ),
            "-c:a", "aac",
            "-shortest",
            "-crf", "18",
            "-preset", "fast",
            out
        ], check=True, capture_output=True)
    
    segment_files.append(out)
    print(f"Segment {i+1}/{len(scenes)}: {out}")

# Create title card
title_card = f"{temp_dir}/title.mp4"
title_filter = (
    f"color=c=#0a0a0a:s={res}:d=5:r={fps},"
    f"drawtext=text='{TOPIC}':fontcolor=white:fontsize=48:x=(w-text_w)/2:y=(h-text_h)/2-30:"
    f"fontfile=/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf,"
    f"drawtext=text='A Documentary':fontcolor=#AAAAAA:fontsize=24:x=(w-text_w)/2:y=(h-text_h)/2+30:"
    f"fontfile=/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
)
subprocess.run([
    "ffmpeg", "-y",
    "-f", "lavfi", "-i", title_filter,
    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-preset", "fast", "-crf", "18",
    title_card
], check=True, capture_output=True)

# Create outro
outro_card = f"{temp_dir}/outro.mp4"
outro_filter = (
    f"color=c=#0a0a0a:s={res}:d=4:r={fps},"
    f"drawtext=text='Thanks for Watching':fontcolor=white:fontsize=42:x=(w-text_w)/2:y=(h-text_h)/2-20:"
    f"fontfile=/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf,"
    f"drawtext=text='Created with Open Source AI':fontcolor=#888888:fontsize=20:x=(w-text_w)/2:y=(h-text_h)/2+30:"
    f"fontfile=/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
)
subprocess.run([
    "ffmpeg", "-y",
    "-f", "lavfi", "-i", outro_filter,
    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-preset", "fast", "-crf", "18",
    outro_card
], check=True, capture_output=True)

# Concatenate all segments
concat_file = f"{temp_dir}/concat.txt"
with open(concat_file, "w") as f:
    for seg in [title_card] + segment_files + [outro_card]:
        abs_path = Path(seg).resolve().as_posix()
        f.write(f"file '{abs_path}'\n")

final_video = f"{final_dir}/documentary.mp4"
subprocess.run([
    "ffmpeg", "-y",
    "-f", "concat", "-safe", "0",
    "-i", concat_file,
    "-c:v", "libx264",
    "-c:a", "aac",
    "-pix_fmt", "yuv420p",
    "-crf", "18",
    "-preset", "medium",
    "-movflags", "+faststart",
    final_video
], check=True, capture_output=True)

import os
size_mb = os.path.getsize(final_video) / 1e6
print(f"\n{'='*50}")
print(f"FINAL VIDEO: {final_video}")
print(f"Size: {size_mb:.1f} MB")
print(f"{'='*50}")

In [ ]:
# Cell 9: Play the video (in notebook)

from IPython.display import Video
Video(final_video, embed=True, width=800)

In [ ]:
# Cell 10: Download zip

import shutil
zip_path = "/kaggle/working/documentary_output.zip"
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', OUTPUT_DIR)
print(f"Download: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")
print("Go to Kaggle sidebar → File → documentary_output.zip → Download")